# Generating subject-verb agreement minimal pairs -- from scratch

This notebook is a **complete, self-contained, simplified** rewrite of
`generate_agreement_minimal_pairs.py` -- every function the original script
had, rewritten here with plainer syntax and lots of comments, so it no
longer depends on that file at all (safe to delete it once you've confirmed
the validation section at the end matches).

Work through the sections top to bottom. Section 5 (marked with 🔧) lets you
build and inspect ONE item at a time, before Section 8 (marked with 🚀) runs
the full generation.

## 1. Basic building blocks

In [ ]:
import random
import json
import pathlib

OPEN_BRACKET = "("
CLOSE_BRACKET = ")"
PLACEHOLDER_VERB = "PLACEHOLDER_VERB"
NUMBER_OF_SWITCHES = 8   # the grammar has 8 word-order switches

### Reading an 8-character language code

**The problem:** every target language (English, Dutch, Japanese, ...) is
defined by an 8-character string like `"01110101"` -- one character per
word-order switch (in order: SV, OV, complementizer position, PP order,
adjective order, relative-clause order, case marking, VP-complement order).
A `"1"` means "flip this switch relative to the base grammar"; a `"0"` means
"leave it as the base grammar has it".

Later on, the function that actually rearranges a sentence
(`flip_word_order`, a few cells down) has to keep asking, as it walks
through the sentence tree: *"is switch number 4 (say) turned on for this
language?"* -- over and over, once per bracket in the tree. Comparing
characters in an 8-character string every time would work fine, but the
**original script represents these 8 yes/no flags as a single integer**
instead, one flag per *bit* of the number -- and this notebook reproduces
that choice so its output matches exactly.

Why bits work for this: a number's binary form is just 8 (or more) yes/no
slots glued together. `174` in binary is `10101110` -- read from the right
(bit 0 first), that's `0,1,1,1,0,1,0,1`, so bits 1, 2, 3, 5, 7 are "on".
This cell's function does the PACKING (string of 0s/1s -> one integer);
`flip_word_order` later does the matching UNPACKING
(`(switch_index >> bit_position) & 1 == 1`) to read each flag back out. The
two are a matched pair -- this one exists purely so the rest of the
notebook has a convenient, pre-packed way to ask "is switch *k* on?".

In [20]:
def lang_id_to_switch_index(lang_id_string):
    """
    lang_id_string is an 8-character string of 0s and 1s, e.g. "01110101".
    Character k (counting from the LEFT, starting at 0) becomes BIT k
    (counting from the RIGHT) of the number we return.
    """
    switch_index = 0
    for position, character in enumerate(lang_id_string):
        if character == "1":
            switch_index += 2 ** position
    return switch_index


# quick check
print('lang_id_to_switch_index("01110101") =', lang_id_to_switch_index("01110101"))
print("(this should be 174 -- matches what the original script always computed for English)")

lang_id_to_switch_index("01110101") = 174
(this should be 174 -- matches what the original script always computed for English)


### Counting how many words exist per category

Before we can pick "a random noun," we need to know how many nouns there
ARE. This reads the grammar file and counts genuine single-word entries for
one category (e.g. `"Noun_S"`), skipping structural rules that happen to
share the same 3-column shape.

In [19]:
def count_lexicon_size(grammar_lines, category_name):
    """
    Count how many different words exist for one category. Only counts
    genuine single-word entries -- a line like
        "0.2   IVerb_Pres_S   IVerb_Pres_S CC IVerb_Pres_S"
    (a coordination rule) has the same category and 3 tab-separated columns
    too, but its "word" is actually 3 tokens with spaces in it, so it must
    be excluded or it would inflate the count by one.
    """
    count = 0
    for line in grammar_lines:
        columns = line.rstrip("\n").split("\t")
        if len(columns) == 3:
            category = columns[1]
            word = columns[2]
            if category == category_name and " " not in word.strip():
                count += 1
    return count

## 2. Building a sentence as a bracketed tree

Every sentence starts life as a tree, written as nested brackets -- e.g.
`"( S ( NP_S ( Noun_S noun_s7 ) ) )"`. Two tiny helper functions build these
trees; everything else in this notebook is built out of just these two.

In [18]:
def make_leaf(category, word):
    """A single word, wrapped in brackets with its category label.
    e.g. make_leaf("Noun_S", "noun_s7") -> ["(", "Noun_S", "noun_s7", ")"]
    """
    return [OPEN_BRACKET, category, word, CLOSE_BRACKET]


def make_node(switch_tag, category, children):
    """
    A structural node that wraps one or more children under a category
    label. `switch_tag` is either None (this rule's order never changes
    between languages) or a number 1-8 (matching one of the grammar's
    word-order switches -- see Section 3).
    """
    if switch_tag is None:
        label = category
    else:
        label = f"{switch_tag}{category}"
    tokens = [OPEN_BRACKET, label]
    for child in children:
        tokens.extend(child)
    tokens.append(CLOSE_BRACKET)
    return tokens


# quick check: build "( S ( NP_S ( Noun_S noun_s7 ) ) )" by hand
example = make_node(None, "S", [make_node(None, "NP_S", [make_leaf("Noun_S", "noun_s7")])])
print(" ".join(example))

( S ( NP_S ( Noun_S noun_s7 ) ) )


## 3. Permuting a tree into a specific language's word order

Every tagged bracket (one whose label starts with a digit, like `"4PP"`)
can be flipped. Its immediate children get reversed in order, if that
switch number is "on" for the target language. This is the exact same
mechanism used to build every training corpus in this whole project.

In [17]:
def reverse_the_top_level_children(tokens):
    """
    Given a list of tokens that represents one or more bracketed "children"
    placed one after another -- e.g. tokens for "(A x)A (B y)B (C z)C" --
    return the SAME children in reversed order: "(C z)C (B y)B (A x)A".
    Anything AFTER the last of those children (the rest of the sentence,
    outside what we're flipping) is left untouched, at the end.
    """
    children = []
    open_positions = []
    for i, token in enumerate(tokens):
        if token == OPEN_BRACKET:
            open_positions.append(i)
        elif token == CLOSE_BRACKET:
            if len(open_positions) == 0:
                # This closing bracket belongs to something OUTSIDE what
                # we're reversing (the parent that contains us) -- stop.
                break
            start = open_positions.pop()
            if len(open_positions) == 0:
                # A complete, self-contained child just finished.
                children.append(tokens[start:i + 1])

    reversed_children_tokens = []
    for child in reversed(children):
        reversed_children_tokens.extend(child)

    everything_after = tokens[len(reversed_children_tokens):]
    return reversed_children_tokens + everything_after


def flip_word_order(switch_index, bracketed_sentence):
    """
    This function does two things, one after another.

    STEP 1 -- figure out which switch numbers are "on".
    switch_index is the single packed number from lang_id_to_switch_index
    (e.g. 174). We just unpack it again: check each of its 8 bits one at a
    time, and every bit that's a 1 means "switch (that bit's position + 1)
    is on". E.g. 174 in binary is 10101110 -- reading from the RIGHT
    (bit 0 first), bits 1, 2, 3, 5, 7 are 1, so switches 2, 3, 4, 6, 8 are
    the ones we need to flip. (Nothing about the sentence is touched yet --
    this step only builds the list of "on" switch numbers.)

    STEP 2 -- walk the sentence and flip every bracket tagged with an "on"
    switch. Concretely: for every tagged bracket like "2VP_S" whose tag
    (here, 2) is in our "on" list, reverse the order of its immediate
    children.

    A tiny worked example: the tree for "the cat chases the dog" might be
    ( 1S ( NP_S cat ) ( 2VP_S ( NP_Obj dog ) ( TVerb chases ) ) )
    With switches 1 and 2 both on:
      - tag 1 ("1S") flips first: [subject, VP] becomes [VP, subject]
      - tag 2 ("2VP_S") flips next: [object, verb] becomes [verb, object]
      - final surface order: "chases dog cat"
    Every switch acts completely independently and only on ITS OWN node's
    direct children -- "1S" never looks inside the VP, "2VP_S" never looks
    at the subject. Whatever final shape the sentence has is just the sum
    of each tagged node independently deciding "keep my children in
    grammar order" or "reverse them", switch by switch.
    """
    switches_to_flip = []
    for bit_position in range(NUMBER_OF_SWITCHES):
        bit_is_on = (switch_index >> bit_position) & 1 == 1
        if bit_is_on:
            switches_to_flip.append(bit_position + 1)

    tokens = bracketed_sentence.split(" ")

    for i in range(len(tokens)):
        first_character = tokens[i][0]
        if first_character.isdigit():
            tag_number = int(first_character)
            if tag_number in switches_to_flip:
                new_ending = reverse_the_top_level_children(tokens[i + 1:])
                tokens = tokens[:i + 1] + new_ending
    return " ".join(tokens)


def strip_brackets(bracketed_sentence):
    """
    Turn "( TAG token1 ( TAG2 token2 )TAG2 )TAG" into "token1 token2 ." --
    every "(" is thrown away along with the tag/label token right after it;
    every ")" is thrown away too. A final "." is always added.
    """
    tokens = bracketed_sentence.split(" ")
    surface_words = []
    i = 0
    while i < len(tokens):
        if tokens[i] == CLOSE_BRACKET:
            i += 1
        elif tokens[i] == OPEN_BRACKET:
            i += 2  # skip the "(" AND the tag/label token right after it
        else:
            surface_words.append(tokens[i])
            i += 1
    surface_words.append(".")
    return " ".join(surface_words)

## 4. Turning lemma indices into real word tokens, and the 4 subject constructions

In [16]:
def the_other_number(number):
    if number == "S":
        return "P"
    else:
        return "S"


def noun_word(number, lemma_index):
    if number == "S":
        return f"noun_s{lemma_index}"
    else:
        return f"noun_p{lemma_index}"


def verb_word(is_transitive, number, lemma_index):
    if is_transitive:
        base = "tverb_pres"
    else:
        base = "iverb_pres"
    if number == "S":
        return f"{base}_s{lemma_index}"
    else:
        return f"{base}_p{lemma_index}"


def verb_category_name(is_transitive, number):
    if is_transitive:
        base = "TVerb_Pres"
    else:
        base = "IVerb_Pres"
    return f"{base}_{number}"

In [15]:
def make_bare_noun_phrase(rng, number, noun_lexicon_size):
    """A noun phrase that's just one bare noun, e.g. 'noun_s7'."""
    lemma_index = rng.randint(1, noun_lexicon_size)
    noun_leaf = make_leaf(f"Noun_{number}", noun_word(number, lemma_index))
    return make_node(None, f"NP_{number}", [noun_leaf])


def make_object_noun_phrase(rng, noun_lexicon_size, number, use_case_markers):
    """
    An object noun phrase. If the grammar marks case (like our English
    setup does, with "sub"/"ob"), it's wrapped with an "ob" marker; some
    grammars (e.g. Dutch) don't mark case at all, so it's skipped there.
    """
    inner_np = make_bare_noun_phrase(rng, number, noun_lexicon_size)
    if not use_case_markers:
        return make_node(None, "NP_Obj", [inner_np])
    ob_marker = make_leaf("Obj", "ob")
    return make_node(7, "NP_Obj", [inner_np, ob_marker])


def wrap_as_subject(number, noun_phrase_node, use_case_markers):
    if not use_case_markers:
        return make_node(None, f"NP_Subj_{number}", [noun_phrase_node])
    sub_marker = make_leaf("Subj", "sub")
    return make_node(7, f"NP_Subj_{number}", [noun_phrase_node, sub_marker])


def make_verb_phrase(rng, number, is_transitive, noun_lexicon_size, verb_leaf, use_case_markers):
    if not is_transitive:
        return make_node(None, f"VP_{number}", [verb_leaf])
    # A transitive verb needs an object; its number doesn't matter for this
    # study, so it's chosen completely at random, just to fill the slot.
    filler_object_number = rng.choice(["S", "P"])
    object_np = make_object_noun_phrase(rng, noun_lexicon_size, filler_object_number, use_case_markers)
    return make_node(2, f"VP_{number}", [object_np, verb_leaf])

### The 4 constructions

Each of these builds the SUBJECT part of one specific construction type.
`"none"` doesn't need its own function, because it is just the `make_bare_noun_phrase`.

In [14]:
def build_subject_pp(rng, number, noun_lexicon_size, preposition_lexicon_size, attractor_number):
    """
    e.g. "the key of the cabinets", where 'key' is the real subject (the head),
    'cabinets' is the preposition's object (the attractor: a second noun
    sitting between the subject and the verb, whose number may or may not
    match the subject's - that mismatch is the whole point of this level).
    """
    head_np = make_bare_noun_phrase(rng, number, noun_lexicon_size)
    attractor_np = make_bare_noun_phrase(rng, attractor_number, noun_lexicon_size)
    preposition_index = rng.randint(1, preposition_lexicon_size)
    preposition_leaf = make_leaf("Prep", f"prep{preposition_index}")
    prepositional_phrase = make_node(4, "PP", [attractor_np, preposition_leaf])
    return make_node(4, f"NP_{number}", [prepositional_phrase, head_np])


def build_subject_rc(rng, number, noun_lexicon_size, verb_lexicon_size, attractor_number, use_case_markers):
    """
    e.g. "the cat that chased the dogs", where 'cat' is both the real subject and (via the relative clause's 
    missing subject) the subject of 'chased' too. 'dogs' is the attractor: the relative clause's OWN
    object, sitting between the real subject and the real (matrix) verb.
    """
    relative_clause_verb_index = rng.randint(1, verb_lexicon_size)
    relative_clause_verb_leaf = make_leaf(
        verb_category_name(True, number), verb_word(True, number, relative_clause_verb_index))
    relative_clause_object = make_object_noun_phrase(rng, noun_lexicon_size, attractor_number, use_case_markers)
    relative_clause_vp = make_node(2, f"VP_{number}", [relative_clause_object, relative_clause_verb_leaf])

    head_noun_index = rng.randint(1, noun_lexicon_size)
    head_noun_leaf = make_leaf(f"Noun_{number}", noun_word(number, head_noun_index))
    relative_word_leaf = make_leaf("Rel", "rel")
    return make_node(6, f"NP_{number}", [relative_clause_vp, relative_word_leaf, head_noun_leaf])


def build_subject_coordination(rng, noun_lexicon_size, closer_conjunct_number):
    """
    e.g. "the cat and the dogs", ALWAYS grammatically plural overall,
    regardless of the two conjuncts' own individual numbers. The second
    (closer-to-the-verb) conjunct's number is what "congruent" refers to.
    """
    first_conjunct_number = rng.choice(["S", "P"])
    first_conjunct = make_bare_noun_phrase(rng, first_conjunct_number, noun_lexicon_size)
    second_conjunct = make_bare_noun_phrase(rng, closer_conjunct_number, noun_lexicon_size)
    and_leaf = make_leaf("CC", "cc")
    return make_node(None, "NP_P", [first_conjunct, and_leaf, second_conjunct])

## 5. Building ONE item 🔧

This is the heart of the whole generator. This function assembles the building blocks into one complete minimal pair. **Change the settings in the next cell and re-run to try different constructions** before moving on to generating the full set.

In [12]:
def build_one_item(rng, level, subject_number, is_transitive, is_congruent,
                    noun_lexicon_size, verb_lexicon_size, preposition_lexicon_size,
                    switch_index, use_case_markers=True):
    """
    Build ONE minimal-pair item: a shared prefix, a correct verb form, a
    wrong (mismatched-number) verb form, and a suffix (if the verb is
    transitive). Returns a plain dictionary with all the fields.
    """
    # Step 1: work out the attractor's number, if this construction has one
    attractor_number = None
    if level == "pp" or level == "rc":
        if is_congruent:
            attractor_number = subject_number
        else:
            attractor_number = the_other_number(subject_number)

    # Step 2: build the SUBJECT part of the sentence (varies by construction)
    if level == "pp":
        subject_node = build_subject_pp(rng, subject_number, noun_lexicon_size,
                                         preposition_lexicon_size, attractor_number)
    elif level == "rc":
        subject_node = build_subject_rc(rng, subject_number, noun_lexicon_size,
                                         verb_lexicon_size, attractor_number, use_case_markers)
    elif level == "coordination":
        if is_congruent:
            closer_conjunct_number = "P"
        else:
            closer_conjunct_number = "S"
        subject_node = build_subject_coordination(rng, noun_lexicon_size, closer_conjunct_number)
    else:  # level == "none"
        subject_node = make_bare_noun_phrase(rng, subject_number, noun_lexicon_size)

    # Step 3: build the VERB PHRASE with a PLACEHOLDER instead of a real verb
    # -- we don't know its final wording yet, and don't need to: word order
    # never depends on what the verb literally says, so the placeholder can
    # safely go through permutation and get replaced afterwards.
    placeholder_leaf = make_leaf(verb_category_name(is_transitive, subject_number), PLACEHOLDER_VERB)
    verb_phrase_node = make_verb_phrase(rng, subject_number, is_transitive, noun_lexicon_size,
                                         placeholder_leaf, use_case_markers)

    # Step 4: assemble the full sentence and permute it into the target
    # language's word order
    subject_with_marker = wrap_as_subject(subject_number, subject_node, use_case_markers)
    whole_sentence_node = make_node(1, "S", [subject_with_marker, verb_phrase_node])
    bracketed_template = " ".join(whole_sentence_node)
    permuted = flip_word_order(switch_index, bracketed_template)
    surface_tokens = strip_brackets(permuted).split(" ")
    surface_tokens.pop()  # drop the trailing ".". It is added back later, per output line

    # Step 5: split the sentence around the placeholder, and fill in the two
    # REAL verb forms with same lemma, but opposite number
    placeholder_position = surface_tokens.index(PLACEHOLDER_VERB)
    prefix_tokens = surface_tokens[:placeholder_position]
    suffix_tokens = surface_tokens[placeholder_position + 1:]

    verb_lemma_index = rng.randint(1, verb_lexicon_size)
    correct_verb = verb_word(is_transitive, subject_number, verb_lemma_index)
    wrong_verb = verb_word(is_transitive, the_other_number(subject_number), verb_lemma_index)

    return {
        "level": level,
        "subject_number": subject_number,
        "transitive": is_transitive,
        "congruent": is_congruent,
        "attractor_number": attractor_number,
        "prefix": " ".join(prefix_tokens),
        "correct_verb": correct_verb,
        "wrong_verb": wrong_verb,
        "suffix": " ".join(suffix_tokens),
    }

### 🔧 Try it: build and inspect one item

Change any of the settings below and re-run this cell to see a different
construction. This is the best place to build intuition before running the
full generation.

In [21]:
GRAMMAR = pathlib.Path("/Users/frapadovani/Desktop/word-order-universals-cogLM/work/grammar/grammar_main/basic-grammar-newlex-zipf-1.gr")

grammar_lines = GRAMMAR.open().readlines()
noun_n = count_lexicon_size(grammar_lines, "Noun_S")
verb_n = count_lexicon_size(grammar_lines, "IVerb_Pres_S")
prep_n = count_lexicon_size(grammar_lines, "Prep")
english_switch_index = lang_id_to_switch_index("01110101")

# ---- change these and re-run ----
try_level = "rc"                # "none", "pp", "rc", or "coordination"
try_subject_number = "S"        # "S" or "P"
try_is_transitive = True        # True or False
try_is_congruent = False        # True, False, or None (only "none" ignores this)
try_seed = 42                   # any integer -- change this to get a DIFFERENT random item

item = build_one_item(random.Random(try_seed), try_level, try_subject_number, try_is_transitive,
                       try_is_congruent, noun_n, verb_n, prep_n, english_switch_index)

print("Full item:")
for key, value in item.items():
    print(f"  {key:16s} {value}")

print()
print("As a sentence:")
print("  correct:", item["prefix"], item["correct_verb"], item["suffix"])
print("  wrong:  ", item["prefix"], item["wrong_verb"], item["suffix"])

Full item:
  level            rc
  subject_number   S
  transitive       True
  congruent        False
  attractor_number P
  prefix           noun_s7 rel tverb_pres_s82 noun_p29 ob sub
  correct_verb     tverb_pres_s29
  wrong_verb       tverb_pres_p29
  suffix           noun_p63 ob

As a sentence:
  correct: noun_s7 rel tverb_pres_s82 noun_p29 ob sub tverb_pres_s29 noun_p63 ob
  wrong:   noun_s7 rel tverb_pres_s82 noun_p29 ob sub tverb_pres_p29 noun_p63 ob


## 6. Turning items into plain sentences, and checking against training data

Before we can guarantee an item is genuinely novel and unseen, we need two things: a way to read an item as a plain list of tokens, and an index of every COMPLETE sentence that actually occurs in the training corpus (or corpora).

In [22]:
def item_as_token_list(item, which_verb):
    """which_verb is either "correct_verb" or "wrong_verb"."""
    tokens = []
    if item["prefix"]:
        tokens.extend(item["prefix"].split(" "))
    tokens.append(item[which_verb])
    if item["suffix"]:
        tokens.extend(item["suffix"].split(" "))
    return tokens


def item_as_sentence_text(item, which_verb):
    """Human-readable form, with a trailing period."""
    return " ".join(item_as_token_list(item, which_verb)) + " ."


def build_training_sentence_lookup(corpus_file_paths, needed_lengths):
    """
    it checks complete sentences, by representing each one as a tuple of all its 
    tokens together, and requiring an exact, in-order match against that entire tuple. 
    """
    sentences_by_length = {}
    for length in needed_lengths:
        sentences_by_length[length] = set()

    for corpus_path in corpus_file_paths:
        with open(corpus_path) as f:
            for line in f:
                tokens = line.rstrip("\n").split(" ")
                if len(tokens) > 0 and tokens[-1] == ".":
                    tokens = tokens[:-1]
                length = len(tokens)
                if length in sentences_by_length:
                    sentences_by_length[length].add(tuple(tokens))
    return sentences_by_length


def item_was_seen_in_training(item, sentences_by_length):
    correct_sentence = tuple(item_as_token_list(item, "correct_verb"))
    wrong_sentence = tuple(item_as_token_list(item, "wrong_verb"))
    correct_seen = correct_sentence in sentences_by_length[len(correct_sentence)]
    wrong_seen = wrong_sentence in sentences_by_length[len(wrong_sentence)]
    return correct_seen or wrong_seen

## 7. Singular/plural pairing

Builds a matched pair (one singular item, one plural item, same lemma indices) by reusing the same random seed for both halves.

In [26]:
def build_matched_singular_and_plural_pair(rng, level, is_transitive, is_congruent,
                                            noun_lexicon_size, verb_lexicon_size, preposition_lexicon_size,
                                            switch_index, use_case_markers, sentences_by_length):
    """
    Build one singular item and one plural item that use the exact same noun/verb/etc lemma indices. 
    If either half turns out to already be present in the training data, 
    both are thrown away and a fresh pair is tried, a pair never survives with only one half kept.
    """
    number_of_rejected_pairs = 0
    while True:
        pair_seed = rng.randrange(2 ** 31)
        singular_item = build_one_item(random.Random(pair_seed), level, "S", is_transitive, is_congruent,
                                        noun_lexicon_size, verb_lexicon_size, preposition_lexicon_size,
                                        switch_index, use_case_markers)
        plural_item = build_one_item(random.Random(pair_seed), level, "P", is_transitive, is_congruent,
                                      noun_lexicon_size, verb_lexicon_size, preposition_lexicon_size,
                                      switch_index, use_case_markers)
        singular_seen = item_was_seen_in_training(singular_item, sentences_by_length)
        plural_seen = item_was_seen_in_training(plural_item, sentences_by_length)
        if singular_seen or plural_seen:
            number_of_rejected_pairs += 1
            continue
        return singular_item, plural_item, number_of_rejected_pairs

## 8. Generating the full set

Every construction type, crossed with transitive/intransitive and congruent/incongruent, `ITEMS_PER_CONDITION` times each, exactly the same grid we traced by hand earlier in this notebook.

In [23]:
ALL_CONDITIONS = (
    # (construction name, which subject numbers exist, does congruent/incongruent apply?)
    ("none", ["S", "P"], False),
    ("pp", ["S", "P"], True),
    ("rc", ["S", "P"], True),
    ("coordination", ["P"], True),  # a coordinated subject is always grammatically plural
)

# ---- settings: change these to control the generation ----
CORPUS = "/Users/frapadovani/Desktop/word-order-universals-cogLM/work/grammar/grammar_main/target_samples_newlex_zipf_1/English_complBefore_01110101.txt"
TRAINING_CORPUS_FILES = [CORPUS, '/Users/frapadovani/Desktop/word-order-universals-cogLM/work/grammar/grammar_main/target_samples_newlex_uniform/English_complBefore_0111010.txt']     # add more paths here if a model trained on several corpora
LANGUAGE_ID = "01110101"             # English
USE_CASE_MARKERS = True
ITEMS_PER_CONDITION = 50
RANDOM_SEED = 455
OUTPUT_JSONL_PATH = "/Users/frapadovani/Desktop/word-order-universals-cogLM/english_agreement_pairs_seed455.jsonl"
OUTPUT_TEXT_PATH = "/Users/frapadovani/Desktop/word-order-universals-cogLM/english_agreement_pairs_seed455.txt"

In [24]:
switch_index = lang_id_to_switch_index(LANGUAGE_ID)
rng = random.Random(RANDOM_SEED)

# Every (level, transitive) combination has a FIXED sentence length,
# regardless of which lemmas get picked -- measure each one once, so the
# training-sentence lookup only needs to cover the lengths we'll actually use.
needed_lengths = set()
for level, subject_numbers, has_congruency in ALL_CONDITIONS:
    for is_transitive in (False, True):
        probe_congruent = True if has_congruency else None
        probe_item = build_one_item(rng, level, subject_numbers[0], is_transitive, probe_congruent,
                                     noun_n, verb_n, prep_n, switch_index, USE_CASE_MARKERS)
        needed_lengths.add(len(item_as_token_list(probe_item, "correct_verb")))

print("Building the training-sentence lookup for lengths", sorted(needed_lengths), "...")
sentences_by_length = build_training_sentence_lookup(TRAINING_CORPUS_FILES, needed_lengths)
print("Done.")

Building the training-sentence lookup for lengths [3, 5, 7, 9] ...
Done.


In [27]:
all_items = []
number_rejected = 0
next_pair_id = 0

for level, subject_numbers, has_congruency in ALL_CONDITIONS:
    for is_transitive in (False, True):
        if has_congruency:
            congruent_values = (True, False)
        else:
            congruent_values = (None,)
        for is_congruent in congruent_values:

            if len(subject_numbers) == 2:
                # a level with both numbers (none/pp/rc) -- build matched pairs
                for _ in range(ITEMS_PER_CONDITION):
                    singular_item, plural_item, n_rejected_here = build_matched_singular_and_plural_pair(
                        rng, level, is_transitive, is_congruent, noun_n, verb_n, prep_n,
                        switch_index, USE_CASE_MARKERS, sentences_by_length)
                    number_rejected += n_rejected_here
                    for item in (singular_item, plural_item):
                        item["pair_id"] = next_pair_id
                        item["id"] = len(all_items)
                        all_items.append(item)
                    next_pair_id += 1
            else:
                # coordination: only one subject number, nothing to pair
                subject_number = subject_numbers[0]
                kept_so_far = 0
                while kept_so_far < ITEMS_PER_CONDITION:
                    item = build_one_item(rng, level, subject_number, is_transitive, is_congruent,
                                           noun_n, verb_n, prep_n, switch_index, USE_CASE_MARKERS)
                    if item_was_seen_in_training(item, sentences_by_length):
                        number_rejected += 1
                        continue
                    item["pair_id"] = None
                    item["id"] = len(all_items)
                    all_items.append(item)
                    kept_so_far += 1

print(f"Built {len(all_items)} items ({number_rejected} training-data collisions discarded along the way)")
for level, _, _ in ALL_CONDITIONS:
    count_for_level = sum(1 for it in all_items if it["level"] == level)
    print(f"  {level:12s} {count_for_level}")

Built 1200 items (21 training-data collisions discarded along the way)
  none         200
  pp           400
  rc           400
  coordination 200


In [28]:
out_jsonl = pathlib.Path(OUTPUT_JSONL_PATH)
out_jsonl.parent.mkdir(parents=True, exist_ok=True)
with out_jsonl.open("w") as f:
    for item in all_items:
        f.write(json.dumps(item) + "\n")
print("Wrote:", out_jsonl)

out_text = pathlib.Path(OUTPUT_TEXT_PATH)
out_text.parent.mkdir(parents=True, exist_ok=True)
with out_text.open("w") as f:
    for item in all_items:
        f.write(item_as_sentence_text(item, "correct_verb") + "\n")
        f.write(item_as_sentence_text(item, "wrong_verb") + "\n")
print("Wrote:", out_text)

Wrote: /Users/frapadovani/Desktop/word-order-universals-cogLM/english_agreement_pairs_seed455.jsonl
Wrote: /Users/frapadovani/Desktop/word-order-universals-cogLM/english_agreement_pairs_seed455.txt
